In [12]:
import os, re, json
# để làm pali-vi, đầu tiên lấy file của pali-vi, ra kết quả (json),
# maping giữa pali-vi và tmc và json của c-pali-tmc-vi (nhớ đổi mn-> mnc)


# Thư mục chứa các file .md",",
SOURCE_DIR = "../../docs/kinhtieubo/thichminhchau"

# Danh sách tên file .md",", cần xử lý (chỉ tên file, không cần đường dẫn đầy đủ)
FILES = [

 {"filename": "kn-001-tap-1-kinh-tieu-tung.md", "range": (1, 10)}
]

NIPATA_TITLES = {
    # "1": "AN 1. Chương Một Pháp",
}

# Heading cấp mấy được coi là "đoạn con" (### = 3, ## = 2, ...)
CHILD_HEADING_LEVEL = 3


## 2. Các hàm xử lý

In [13]:
TOP_INDEX_RE = re.compile(r'^[a-z]+-0*(\d+)')
H1_RE = re.compile(r'^#\s+(.*)$', re.MULTILINE)
CHILD_RE = re.compile(r'^#{%d}\s+(.*)$' % CHILD_HEADING_LEVEL)
NUM_PREFIX_RE = re.compile(r'^(\d+(?:\.\d+)+)\.?\s*')


import re
import unicodedata

def slugify(text):
    """Đoán anchor kiểu markdown-it-anchor (đã hỗ trợ bỏ dấu tiếng Việt)."""
    text = text.strip()
    text = re.sub(r'`([^`]*)`', r'\1', text)      # bỏ backtick code

    # 1. Xử lý riêng chữ đ/Đ vì unicodedata không tự chuyển thành d
    text = text.replace('đ', 'd').replace('Đ', 'D')

    # 2. Tách dấu ra khỏi ký tự và loại bỏ chúng
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

    # 3. Chuẩn hóa slug
    text = text.lower()
    text = re.sub(r'[^\w\s-]', '', text)          # Xóa các ký tự đặc biệt (chấm, phẩy, !, ?) thay vì biến thành gạch ngang
    text = re.sub(r'[\s_]+', '-', text)           # Đổi khoảng trắng/gạch dưới thành gạch ngang
    text = re.sub(r'-+', '-', text)               # Gộp nhiều gạch ngang liên tiếp thành 1

    return text.strip('-')

def top_index_from_slug(slug):
    m = TOP_INDEX_RE.match(slug.lower())
    return m.group(1) if m else None


def extract_h1(text):
    m = H1_RE.search(text)
    return m.group(1).strip() if m else None


def insert_path(item, path, default_slug, slug=None, anchor=None):
    """Đi xuống item['children'][...] theo path (list số dạng string), tạo node
    nếu chưa có. Ở node lá: chỉ gắn slug nếu KHÁC trang mặc định của item (tránh
    lặp thừa khi con nằm cùng trang cha)."""
    node = item
    for i, k in enumerate(path):
        node.setdefault("children", {})
        node["children"].setdefault(k, {})
        node = node["children"][k]
        if i == len(path) - 1:
            if slug and slug != default_slug:
                node["slug"] = slug
            if anchor:
                node["anchor"] = anchor


def process_file(dirpath, spec, items, warnings, titles_override):
    filename = spec["filename"]
    file_range = spec.get("range")  # (start, end) hoặc None

    slug = filename[:-3] if filename.endswith(".md") else filename
    filepath = os.path.join(dirpath, filename)
    if not os.path.isfile(filepath):
        warnings.append(f"{filename}: không tìm thấy file, bỏ qua")
        return

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    top_index = top_index_from_slug(slug)
    if top_index is None:
        warnings.append(f"{filename}: không suy ra được số thứ tự (top index) từ tên file, bỏ qua")
        return
    key = str(int(top_index))

    is_new_key = key not in items
    item = items.setdefault(key, {})

    if is_new_key:
        if key in titles_override:
            item["title"] = titles_override[key]
        else:
            h1 = extract_h1(text)
            item["title"] = h1 if h1 else f"??? (chưa có tiêu đề cho key {key})"
            if not h1:
                warnings.append(f"{filename}: không tìm thấy H1, cần điền title tay cho key {key}")
        # Trang mặc định khi user chỉ gõ tới top-index (vd "an 1"), không có số con.
        # Với case gộp nhiều file, đây là trang của file ĐẦU TIÊN gặp trong danh sách FILES.
        item["slug"] = slug
    elif key in titles_override and item.get("title") != titles_override[key]:
        item["title"] = titles_override[key]

    default_slug = item["slug"]

    # 1) điền mặc định theo range khai báo -> đảm bảo MỌI số trong range có link,
    #    kể cả khi trong file không có heading riêng cho từng kinh
    if file_range:
        start, end = file_range
        for n in range(start, end + 1):
            insert_path(item, [str(n)], default_slug, slug=slug)

    # 2) quét heading đánh số trong file để bổ sung anchor chính xác (nếu có)
    for line in text.splitlines():
        m = CHILD_RE.match(line)
        if not m:
            continue
        heading_text = m.group(1).strip()
        num_match = NUM_PREFIX_RE.match(heading_text)
        if not num_match:
            continue
        numbers = num_match.group(1).split(".")
        if numbers[0] != key:
            warnings.append(
                f'{filename}: heading "{heading_text}" có số đầu ({numbers[0]}) khác key ({key})'
            )
            continue
        path = numbers[1:]
        if not path:
            continue
        anchor = slugify(heading_text)
        insert_path(item, path, default_slug, slug=slug, anchor=anchor)


## 3. Chạy xử lý

In [14]:
items = {}
warnings = []

for fn in FILES:
    process_file(SOURCE_DIR, fn, items, warnings, NIPATA_TITLES)

if warnings:
    print("⚠️  Cảnh báo:")
    for w in warnings:
        print(" -", w)
else:
    print("Không có cảnh báo.")


Không có cảnh báo.


## 4. Kết quả — dán vào `quicklink-data.js`

Các trường `"???"` là chỗ bạn tự điền (`folder`, edition key, `label`, `path`, `index_length`).

In [15]:
output = {
    "folder": "kinhtieubo",
    "editions": {
        "tmc": {
            "label": "Pali (Vi)",
            "path": "pali-vi",
            "index_length": "3",
            "items": items,
        }
    },
}

print(json.dumps(output, ensure_ascii=False, indent=2))


{
  "folder": "kinhtieubo",
  "editions": {
    "tmc": {
      "label": "Pali (Vi)",
      "path": "pali-vi",
      "index_length": "3",
      "items": {
        "1": {
          "title": "TẬP 1 – KINH TIỂU TỤNG",
          "slug": "kn-001-tap-1-kinh-tieu-tung",
          "children": {
            "1": {},
            "2": {},
            "3": {},
            "4": {},
            "5": {},
            "6": {},
            "7": {},
            "8": {},
            "9": {},
            "10": {}
          }
        }
      }
    }
  }
}


## 5. (Tùy chọn) Ghi ra file JSON

Chạy cell dưới nếu muốn lưu kết quả ra file thay vì chỉ copy từ output ở trên.

In [ ]:
OUT_PATH = "quicklink-data.generated.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Đã ghi: {OUT_PATH}")
